In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import time
from urllib.parse import urljoin
import pandas as pd
import sqlite3

In [4]:
BASE_URL = "https://books.toscrape.com/"

# 6 categories comfortably clears the >=60 book / >=3 category requirement
CATEGORIES = {
    "Travel": "catalogue/category/books/travel_2/index.html",
    "Mystery": "catalogue/category/books/mystery_3/index.html",
    "Historical Fiction": "catalogue/category/books/historical-fiction_4/index.html",
    "Classics": "catalogue/category/books/classics_6/index.html",
    "Philosophy": "catalogue/category/books/philosophy_7/index.html",
    "Romance": "catalogue/category/books/romance_8/index.html",
}

RATING_WORDS = {"One", "Two", "Three", "Four", "Five"}

In [5]:
all_books = []

for category_name, start_path in CATEGORIES.items():
    page_url = urljoin(BASE_URL, start_path)

    while page_url:
        response = requests.get(page_url, timeout=10)
        response.encoding = "utf-8"  # books.toscrape.com sometimes gets mis-detected as latin-1,
                                      # which turns "£" into "Ã‚£" (mojibake) in response.text
        soup = BeautifulSoup(response.text, "html.parser")

        for article in soup.select("article.product_pod"):
            title = article.h3.a["title"]
            price_text = article.select_one("p.price_color").get_text(strip=True)

            # star rating isn't in any text -- it's a CSS class like "star-rating Three"
            rating_tag = article.select_one("p.star-rating")
            rating_word = None
            for css_class in rating_tag.get("class", []):
                if css_class in RATING_WORDS:
                    rating_word = css_class

            availability_text = article.select_one("p.instock.availability").get_text(strip=True)

            all_books.append({
                "title": title,
                "price": price_text,
                "star_rating": rating_word,
                "availability": availability_text,
                "category": category_name
            })

        next_link = soup.select_one("li.next a")
        page_url = urljoin(page_url, next_link["href"]) if next_link else None
        time.sleep(0.3)

print(f"Scraped {len(all_books)} books")

Scraped 134 books


In [7]:
df = pd.DataFrame(all_books)
df.to_csv("books_raw.csv", index=False)
df

,title,price,star_rating,availability,category
0,It's Only the Himalayas,£45.17,Two,In stock,Travel
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,£37.33,Three,In stock,Travel
...,...,...,...,...,...
129,Imperfect Harmony,£34.74,Four,In stock,Romance
130,Fighting Fate (Fighting #6),£39.24,Three,In stock,Romance
131,Deep Under (Walker Security #1),£47.09,Five,In stock,Romance
132,Charity's Cross (Charles Towne Belles #4),£41.24,One,In stock,Romance


In [8]:
df = pd.read_csv("books_raw.csv")
df.isnull().sum()

title           0
price           0
star_rating     0
availability    0
category        0
dtype: int64

In [9]:

import re


df['price_gbp'] = df['price'].apply(lambda s: re.sub(r'[^\d.]', '', str(s)))
df['price_gbp'] = pd.to_numeric(df['price_gbp'], errors='coerce')

In [10]:
rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
df['rating'] = df['star_rating'].map(rating_map)

In [11]:
df['in_stock'] = df['availability'].str.contains('In stock', case=False, na=False)

In [12]:
df.isnull().sum()

title           0
price           0
star_rating     0
availability    0
category        0
price_gbp       0
rating          0
in_stock        0
dtype: int64

In [13]:
median = df['price_gbp'].median()
df['price_gbp'] = df['price_gbp'].fillna(median)
print(f"Median-imputed price_gbp with {median:.2f}")

Median-imputed price_gbp with 33.47


In [14]:
n_before = len(df)
df = df.dropna(subset=['rating'])
df['rating'] = df['rating'].astype(int)
print(f"Dropped {n_before - len(df)} row(s) with an unparseable rating")

Dropped 0 row(s) with an unparseable rating


In [15]:
df['price_inr'] = df['price_gbp'] * 105.50

In [16]:
df = df[['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category']]
df.to_csv("books_clean.csv", index=False)
df

,title,price_gbp,price_inr,rating,in_stock,category
0,It's Only the Himalayas,45.17,4765.435,2,True,Travel
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,49.43,5214.865,4,True,Travel
2,See America: A Celebration of Our National Par...,48.87,5155.785,3,True,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.170,2,True,Travel
4,Under the Tuscan Sun,37.33,3938.315,3,True,Travel
...,...,...,...,...,...,...
129,Imperfect Harmony,34.74,3665.070,4,True,Romance
130,Fighting Fate (Fighting #6),39.24,4139.820,3,True,Romance
131,Deep Under (Walker Security #1),47.09,4967.995,5,True,Romance
132,Charity's Cross (Charles Towne Belles #4),41.24,4350.820,1,True,Romance


In [17]:
conn = sqlite3.connect("zepto_catalog.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")

conn.commit()

In [19]:
df = pd.read_csv("books_clean.csv")

category_names = df["category"].unique().tolist()

cursor.executemany(
    """
    INSERT OR IGNORE INTO categories (category_name)
    VALUES (?)
    """,
    [(name,) for name in category_names]
)

conn.commit()

print("Categories inserted successfully.")

Categories inserted successfully.


In [20]:
name_to_id = dict(cursor.execute("SELECT category_name, category_id FROM categories").fetchall())

book_rows = []
for row in df.itertuples(index=False):
    book_rows.append(
        (row.title, row.price_gbp, row.price_inr, row.rating, int(row.in_stock), name_to_id[row.category])
    )

cursor.executemany(
    "INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id) VALUES (?, ?, ?, ?, ?, ?)",
    book_rows
)
conn.commit()
conn.close()

print(f"Loaded {len(book_rows)} books across {len(category_names)} categories")

Loaded 134 books across 6 categories


In [21]:
conn = sqlite3.connect("zepto_catalog.db")

query = """
SELECT title, rating, price_inr
FROM books
WHERE rating >= 4 AND in_stock = 1
ORDER BY price_inr DESC
LIMIT 10
"""
result = pd.read_sql(query, conn)
conn.close()
result

,title,rating,price_inr
0,The Death of Humanity: and the Case for Life,4,6130.605
1,The Death of Humanity: and the Case for Life,4,6130.605
2,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,4,6087.350
3,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,4,6087.350
4,A Year in Provence (Provence #1),4,6000.840
5,A Year in Provence (Provence #1),4,6000.840
6,The Past Never Ends,4,5960.750
7,The Past Never Ends,4,5960.750
8,A Flight of Arrows (The Pathfinders #2),5,5858.415
9,A Flight of Arrows (The Pathfinders #2),5,5858.415


In [22]:
conn = sqlite3.connect("zepto_catalog.db")

query = "SELECT DISTINCT category_name FROM categories"
result = pd.read_sql(query, conn)
conn.close()
result

,category_name
0,Travel
1,Mystery
2,Historical Fiction
3,Classics
4,Philosophy
5,Romance


In [23]:
conn = sqlite3.connect("zepto_catalog.db")

query = """
SELECT title, price_gbp
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp ASC
"""
result = pd.read_sql(query, conn)
conn.close()
result

,title,price_gbp
0,"Run, Spot, Run: The Ethics of Keeping Pets",20.02
1,"Run, Spot, Run: The Ethics of Keeping Pets",20.02
2,Blood Defense (Samantha Brinkman #1),20.30
3,Blood Defense (Samantha Brinkman #1),20.30
4,"Love, Lies and Spies",20.55
...,...,...
123,A Paris Apartment,39.01
124,Fighting Fate (Fighting #6),39.24
125,Fighting Fate (Fighting #6),39.24
126,Where Lightning Strikes (Bleeding Stars #3),39.77


In [24]:
conn = sqlite3.connect("zepto_catalog.db")

query = """
SELECT b.title, c.category_name
FROM books AS b
JOIN categories AS c ON b.category_id = c.category_id
WHERE c.category_name IN ('Travel', 'Mystery', 'Classics')
ORDER BY c.category_name, b.title
"""
result = pd.read_sql(query, conn)
conn.close()
result

,title,category_name
0,Alice in Wonderland (Alice's Adventures in Won...,Classics
1,Alice in Wonderland (Alice's Adventures in Won...,Classics
2,And Then There Were None,Classics
3,And Then There Were None,Classics
4,Animal Farm,Classics
...,...,...
119,The Road to Little Dribbling: Adventures of an...,Travel
120,Under the Tuscan Sun,Travel
121,Under the Tuscan Sun,Travel
122,Vagabonding: An Uncommon Guide to the Art of L...,Travel


In [25]:
join_query = """
SELECT
    c.category_name,
    b.title,
    b.rating,
    b.price_inr
FROM books AS b
JOIN categories AS c ON b.category_id = c.category_id
WHERE b.rating = (
    SELECT MAX(rating) FROM books WHERE category_id = b.category_id
)
ORDER BY c.category_name, b.title
"""

conn = sqlite3.connect("zepto_catalog.db")
df_join_sql = pd.read_sql(join_query, conn)
df_join_sql

,category_name,title,rating,price_inr
0,Classics,Little Women (Little Women #1),4,2961.385
1,Classics,Little Women (Little Women #1),4,2961.385
2,Classics,The Complete Stories and Poems (The Works of E...,4,2825.290
3,Classics,The Complete Stories and Poems (The Works of E...,4,2825.290
4,Classics,The Secret Garden,4,1590.940
5,Classics,The Secret Garden,4,1590.940
6,Classics,The Story of Hong Gildong,4,4556.545
7,Classics,The Story of Hong Gildong,4,4556.545
8,Historical Fiction,A Flight of Arrows (The Pathfinders #2),5,5858.415
9,Historical Fiction,A Flight of Arrows (The Pathfinders #2),5,5858.415


In [26]:
result1 = pd.read_sql("""
SELECT title, rating, price_inr
FROM books
WHERE rating >= 4 AND in_stock = 1
ORDER BY price_inr DESC
LIMIT 10
""", conn)
result1

,title,rating,price_inr
0,The Death of Humanity: and the Case for Life,4,6130.605
1,The Death of Humanity: and the Case for Life,4,6130.605
2,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,4,6087.350
3,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,4,6087.350
4,A Year in Provence (Provence #1),4,6000.840
5,A Year in Provence (Provence #1),4,6000.840
6,The Past Never Ends,4,5960.750
7,The Past Never Ends,4,5960.750
8,A Flight of Arrows (The Pathfinders #2),5,5858.415
9,A Flight of Arrows (The Pathfinders #2),5,5858.415


In [27]:
books_df = pd.read_sql("SELECT * FROM books", conn)
categories_df = pd.read_sql("SELECT * FROM categories", conn)
conn.close()

merged = pd.merge(books_df, categories_df, on='category_id', how='inner')

max_rating = merged.groupby('category_name')['rating'].transform('max')
df_join_merge = merged[merged['rating'] == max_rating][['category_name', 'title', 'rating', 'price_inr']]
df_join_merge = df_join_merge.sort_values(['category_name', 'title']).reset_index(drop=True)
df_join_merge

,category_name,title,rating,price_inr
0,Classics,Little Women (Little Women #1),4,2961.385
1,Classics,Little Women (Little Women #1),4,2961.385
2,Classics,The Complete Stories and Poems (The Works of E...,4,2825.290
3,Classics,The Complete Stories and Poems (The Works of E...,4,2825.290
4,Classics,The Secret Garden,4,1590.940
5,Classics,The Secret Garden,4,1590.940
6,Classics,The Story of Hong Gildong,4,4556.545
7,Classics,The Story of Hong Gildong,4,4556.545
8,Historical Fiction,A Flight of Arrows (The Pathfinders #2),5,5858.415
9,Historical Fiction,A Flight of Arrows (The Pathfinders #2),5,5858.415


In [28]:
print(f"SQL JOIN rows: {len(df_join_sql)}")
print(f"pd.merge rows: {len(df_join_merge)}")

SQL JOIN rows: 48
pd.merge rows: 48


In [29]:
comparison = pd.concat(
    [
        df_join_sql.reset_index(drop=True).add_prefix("SQL_"),
        df_join_merge.reset_index(drop=True).add_prefix("MERGE_")
    ],
    axis=1
)

display(comparison)

print("Results equivalent:",
      df_join_sql.reset_index(drop=True).equals(
          df_join_merge.reset_index(drop=True)
      ))

,SQL_category_name,SQL_title,SQL_rating,SQL_price_inr,MERGE_category_name,MERGE_title,MERGE_rating,MERGE_price_inr
0,Classics,Little Women (Little Women #1),4,2961.385,Classics,Little Women (Little Women #1),4,2961.385
1,Classics,Little Women (Little Women #1),4,2961.385,Classics,Little Women (Little Women #1),4,2961.385
2,Classics,The Complete Stories and Poems (The Works of E...,4,2825.290,Classics,The Complete Stories and Poems (The Works of E...,4,2825.290
3,Classics,The Complete Stories and Poems (The Works of E...,4,2825.290,Classics,The Complete Stories and Poems (The Works of E...,4,2825.290
4,Classics,The Secret Garden,4,1590.940,Classics,The Secret Garden,4,1590.940
5,Classics,The Secret Garden,4,1590.940,Classics,The Secret Garden,4,1590.940
6,Classics,The Story of Hong Gildong,4,4556.545,Classics,The Story of Hong Gildong,4,4556.545
7,Classics,The Story of Hong Gildong,4,4556.545,Classics,The Story of Hong Gildong,4,4556.545
8,Historical Fiction,A Flight of Arrows (The Pathfinders #2),5,5858.415,Historical Fiction,A Flight of Arrows (The Pathfinders #2),5,5858.415
9,Historical Fiction,A Flight of Arrows (The Pathfinders #2),5,5858.415,Historical Fiction,A Flight of Arrows (The Pathfinders #2),5,5858.415


Results equivalent: True


In [30]:
df["price_inr"] = (df["price_gbp"] * 105.50).round(2)

In [31]:
price_check = (
    df["price_inr"].round(2)
    == (df["price_gbp"] * 105.50).round(2)
).all()

print("Price conversion correct:", price_check)

Price conversion correct: True


In [32]:
print("Final rows:", len(df))
print("Categories:", df["category"].nunique())
print("price_gbp dtype:", df["price_gbp"].dtype)
print("price_inr dtype:", df["price_inr"].dtype)
print("rating dtype:", df["rating"].dtype)
print("in_stock dtype:", df["in_stock"].dtype)

Final rows: 134
Categories: 6
price_gbp dtype: float64
price_inr dtype: float64
rating dtype: int64
in_stock dtype: bool
